# 02 — Built-up Classification via NDBI

**Goal:** turn the proven Nairobi composite (notebook 01) into a binary built-up/non-built-up classification, and measure how good it actually is against an independent reference (ESA WorldCover) rather than eyeballing it.

Baseline approach: NDBI (Normalized Difference Built-up Index) thresholded at 0 — the simplest possible classifier. This establishes a real, measured baseline number before any tuning or a trained model is considered.

In [1]:
import sys
sys.path.insert(0, '../src')

import ee
import geemap
import urllib.request
from acquisition import get_nairobi_boundary, get_sentinel2_composite

ee.Initialize(project='solar-haven-349708')

nairobi = get_nairobi_boundary()
composite, scene_count = get_sentinel2_composite(
    nairobi, start_date='2024-06-01', end_date='2024-09-30', cloud_threshold=20
)
print(f'Composite ready: {scene_count} scenes')

Composite ready: 11 scenes


## NDBI and threshold

`NDBI = (SWIR1 - NIR) / (SWIR1 + NIR)`, using Sentinel-2 bands B11 (SWIR1) and B8 (NIR). Built-up surfaces (concrete, roofing, bare ground) reflect relatively more SWIR than NIR, so they trend toward higher NDBI; vegetation and water trend lower. `ee.Image.normalizedDifference(['B11', 'B8'])` computes this directly.

Threshold: NDBI > 0. This is the textbook default, not tuned to Nairobi specifically — the point of this notebook is to measure how well that untuned default actually performs.

In [2]:
ndbi = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')
ndbi_builtup = ndbi.gt(0).rename('builtup')

## Reference: ESA WorldCover

ESA WorldCover 2021 (`ESA/WorldCover/v200`, 10m resolution) is an independently produced, published global land-cover map — not ground truth, but a real external reference rather than a number we made up. Class value 50 in its `Map` band is "Built-up." We pull the same class for Nairobi and compare pixel-by-pixel against our NDBI classification.

In [3]:
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(nairobi)
worldcover_builtup = worldcover.eq(50).rename('builtup')

agreement_image = ndbi_builtup.eq(worldcover_builtup)
agreement_stats = agreement_image.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=nairobi,
    scale=10,
    maxPixels=1e9,
)
agreement_pct = agreement_stats.getInfo()['builtup'] * 100
print(f'Pixel agreement with WorldCover built-up class: {agreement_pct:.1f}%')

Pixel agreement with WorldCover built-up class: 62.0%


## Diagnosing the disagreement

A single agreement number doesn't say *how* the two methods disagree. Checking each method's built-up area fraction separately shows whether NDBI is systematically over- or under-classifying relative to WorldCover, which is more actionable than the agreement percentage alone.

In [4]:
ndbi_frac = ndbi_builtup.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
).getInfo()['builtup'] * 100

wc_frac = worldcover_builtup.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
).getInfo()['builtup'] * 100

print(f'NDBI-classified built-up: {ndbi_frac:.1f}% of Nairobi')
print(f'WorldCover-classified built-up: {wc_frac:.1f}% of Nairobi')

NDBI-classified built-up: 56.4% of Nairobi
WorldCover-classified built-up: 31.9% of Nairobi


## Visualize

Interactive comparison map (live in Jupyter), plus a static thumbnail of the NDBI classification alone for a quick no-Jupyter check.

In [5]:
builtup_vis = {'min': 0, 'max': 1, 'palette': ['black', 'red']}

Map = geemap.Map(center=[-1.290, 36.868], zoom=11)
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'True color', False)
Map.addLayer(ndbi_builtup, builtup_vis, 'NDBI built-up (this notebook)')
Map.addLayer(worldcover_builtup, builtup_vis, 'WorldCover built-up (reference)', False)
Map.addLayerControl()
Map

Map(center=[-1.29, 36.868], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [6]:
for name, image in [('ndbi', ndbi_builtup), ('worldcover', worldcover_builtup)]:
    url = image.getThumbURL({
        'min': 0, 'max': 1, 'palette': ['black', 'red'],
        'region': nairobi, 'dimensions': 800,
    })
    path = f'../data/processed/nairobi_builtup_{name}.png'
    urllib.request.urlretrieve(url, path)
    print(f'Saved {path}')

Saved ../data/processed/nairobi_builtup_ndbi.png


Saved ../data/processed/nairobi_builtup_worldcover.png
